In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
import sys
!{sys.executable} -m pip install seaborn



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
"c:\Program" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [3]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [4]:
import pandas as pd
import numpy as np
#import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
import pandas as pd
import awswrangler as wr

# Parámetros
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Ruta S3 del parquet
s3_path = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA/data_pn_total_expandido_new.parquet"

# Cargar parquet desde S3
df_inference = wr.s3.read_parquet(path=s3_path, boto3_session=session)

print(f"✓ Parquet cargado desde S3")
print(f"  Shape: {df_inference.shape}")
print(f"  Columnas: {df_inference.columns.tolist()[:10]}...")  # primeras 10
display(df_inference.head())

✓ Parquet cargado desde S3
  Shape: (1684824, 65)
  Columnas: ['target', 'key_value', 'cod_cli', 'codmes_lag1', 'cod_mes', 'mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'max_trx_abonos_3m']...


,target,key_value,cod_cli,codmes_lag1,cod_mes,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,share_cp_ingresos,ros_por_trx_3m,alertas_por_antiguedad,ingresos_vs_facturacion,pasivo_vs_ingresos,ratio_egresos_exterior,ratio_ingresos_exterior,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente
0,0,613CFE1D86595756D0BD26EBF1C389FFF338DA6B92CA68...,0016324312,202502,202503,51804.35,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,SIN_INFO
1,0,48D2AFCEB9D20B54584C4BDE979F83EC17382A05A45448...,0017895869,202505,202506,293814.45,24180.00,0.00,0.00,3770.00,...,0.00,0.00,0.00,0.00,8.41,0.00,0.00,0.00,0,SIN_INFO
2,0,3090515E4CA50B9BD515B6CF718A0090BE40F45F79F256...,0020574907,202502,202503,0.65,0.00,0.00,0.00,4400.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,SIN_INFO
3,0,05B0B55EBE48978C9810AB4CAF436807FAAC99852244B7...,0018661705,202505,202506,13574.85,14955.00,400.00,0.33,30410.00,...,0.00,0.00,0.00,0.00,0.15,0.00,0.00,0.00,0,SIN_INFO
4,0,64C3B69BEAB279FEDF577E66488F0AFEFB2C613C3B3609...,0019930894,202412,202501,4199.80,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,SIN_INFO


In [6]:
df_inference.shape

(1684824, 65)

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL   = "target"
PERIODO_COL  = "cod_mes"
PERIODOS     = [202508, 202509, 202510, 202511, 202601, 202602, 202603,202604]
OUTPUT_DIR   = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\bivariados_graficos_test"
N_BINS       = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Variables seleccionadas ──────────────────────────────────────────────────
VARIABLES_SELECCIONADAS = [
    'mto_pas_soles',
 'imp_trx_abonosefect_6m',
 'imp_trx_cargosefe_6m',
 'avg_trx_cargostot_3m',
 'cnt_trx_cargostot_3m',
 'cnt_trx_abonospromtot_3m',
 'rat_trx_abonosefectot_1m',
 'rat_trx_abonosefectot_3m',
 'rat_trx_abonosefectot_9m',
 'rat_mntcrgsefetot_1m',
 'num_antiguedad',
 'cnt_meses_sinegresos_12m',
 'cod_ubigeo_cd',
 'cod_sectorista_id',
 'flg_vrcn_abonos_5m_1m',
 'flg_vrcn_efe_cargos_5m_1m',
 'mto_fact_declarado_sunat',
 'avg_cp_men_ing_12m',
 'avg_cpmenegr_12m',
 'max_mto_cpegrmen_12m',
 'cnt_trx_sinenv_alext_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'cnt_noticias',
 'flg_alerta_12m',
 'cnt_alerta_hist',
 'cnt_ros_hist',
 'ratio_abonos_1m_vs_6m',
 'ratio_cargos_1m_vs_6m',
 'share_cp_egresos',
 'share_cp_ingresos',
 'ros_por_trx_3m',
 'ingresos_vs_facturacion',
 'pasivo_vs_ingresos',
 'ratio_egresos_exterior',
 'ratio_ingresos_exterior',
]

# ── Filtrar periodos ─────────────────────────────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
print(f"✓ Registros filtrados: {df.shape[0]:,}  |  Target rate: {df[TARGET_COL].mean():.4f}")

# ── Validar columnas existentes ──────────────────────────────────────────────
variables     = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas en el DataFrame: {no_existentes}")
print(f"✓ Variables a graficar: {len(variables)}")

# ── Función de análisis bivariado ────────────────────────────────────────────
def plot_bivariado(df, col, target, n_bins, output_dir):
    df_tmp = df[[col, target]].copy()
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

    try:
        df_tmp["bin"] = pd.qcut(df_tmp[col], q=n_bins, duplicates="drop")
    except Exception:
        df_tmp["bin"] = pd.cut(df_tmp[col], bins=n_bins)

    resumen = (
        df_tmp.groupby("bin", observed=True)[target]
        .agg(count="count", target_rate="mean")
        .reset_index()
    )
    resumen["bin_str"] = resumen["bin"].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.bar(resumen["bin_str"], resumen["count"], color="#4C72B0", alpha=0.7, label="Registros")
    ax1.set_xlabel(col, fontsize=11)
    ax1.set_ylabel("Registros", fontsize=11, color="#4C72B0")
    ax1.tick_params(axis="x", rotation=45, labelsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    ax2 = ax1.twinx()
    ax2.plot(resumen["bin_str"], resumen["target_rate"], color="#DD4949",
             marker="o", linewidth=2, label="Target rate")
    ax2.set_ylabel("Target rate", fontsize=11, color="#DD4949")
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    plt.title(f"Bivariado: {col}  |  periodos {PERIODOS[0]}–{PERIODOS[-1]}", fontsize=13)
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"biv_{col}.png")
    plt.savefig(filepath, dpi=120, bbox_inches="tight")
    plt.close()
    return filepath

# ── Loop principal ───────────────────────────────────────────────────────────
guardados = []
errores   = []

for i, col in enumerate(variables, 1):
    try:
        fp = plot_bivariado(df, col, TARGET_COL, N_BINS, OUTPUT_DIR)
        guardados.append(fp)
        print(f"  [{i}/{len(variables)}] {col} ✓")
    except Exception as e:
        errores.append((col, str(e)))

print(f"\n✅ Gráficos guardados: {len(guardados)}")
print(f"⚠  Errores:           {len(errores)}")
if errores:
    for col, err in errores:
        print(f"   → {col}: {err}")
print(f"\n📁 Carpeta: {OUTPUT_DIR}")

✓ Registros filtrados: 1,338,493  |  Target rate: 0.0008
✓ Variables a graficar: 36
  [1/36] mto_pas_soles ✓
  [2/36] imp_trx_abonosefect_6m ✓
  [3/36] imp_trx_cargosefe_6m ✓
  [4/36] avg_trx_cargostot_3m ✓
  [5/36] cnt_trx_cargostot_3m ✓
  [6/36] cnt_trx_abonospromtot_3m ✓
  [7/36] rat_trx_abonosefectot_1m ✓
  [8/36] rat_trx_abonosefectot_3m ✓
  [9/36] rat_trx_abonosefectot_9m ✓
  [10/36] rat_mntcrgsefetot_1m ✓
  [11/36] num_antiguedad ✓
  [12/36] cnt_meses_sinegresos_12m ✓
  [13/36] cod_ubigeo_cd ✓
  [14/36] cod_sectorista_id ✓
  [15/36] flg_vrcn_abonos_5m_1m ✓
  [16/36] flg_vrcn_efe_cargos_5m_1m ✓
  [17/36] mto_fact_declarado_sunat ✓
  [18/36] avg_cp_men_ing_12m ✓
  [19/36] avg_cpmenegr_12m ✓
  [20/36] max_mto_cpegrmen_12m ✓
  [21/36] cnt_trx_sinenv_alext_12m ✓
  [22/36] mto_al_ext_12m ✓
  [23/36] mto_del_ext_12m ✓
  [24/36] cnt_noticias ✓
  [25/36] flg_alerta_12m ✓
  [26/36] cnt_alerta_hist ✓
  [27/36] cnt_ros_hist ✓
  [28/36] ratio_abonos_1m_vs_6m ✓
  [29/36] ratio_cargos_1m_vs_6m

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
#import seaborn as sns
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL  = "target"
PERIODO_COL = "cod_mes"
PERIODOS    = [202508, 202509, 202510, 202511, 202601, 202602, 202603,202604]
OUTPUT_DIR  = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\eda_graficos_test"
N_BINS      = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

VARIABLES_SELECCIONADAS = [
    'mto_pas_soles',
 'imp_trx_abonosefect_6m',
 'imp_trx_cargosefe_6m',
 'avg_trx_cargostot_3m',
 'cnt_trx_cargostot_3m',
 'cnt_trx_abonospromtot_3m',
 'rat_trx_abonosefectot_1m',
 'rat_trx_abonosefectot_3m',
 'rat_trx_abonosefectot_9m',
 'rat_mntcrgsefetot_1m',
 'num_antiguedad',
 'cnt_meses_sinegresos_12m',
 'cod_ubigeo_cd',
 'cod_sectorista_id',
 'flg_vrcn_abonos_5m_1m',
 'flg_vrcn_efe_cargos_5m_1m',
 'mto_fact_declarado_sunat',
 'avg_cp_men_ing_12m',
 'avg_cpmenegr_12m',
 'max_mto_cpegrmen_12m',
 'cnt_trx_sinenv_alext_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'cnt_noticias',
 'flg_alerta_12m',
 'cnt_alerta_hist',
 'cnt_ros_hist',
 'ratio_abonos_1m_vs_6m',
 'ratio_cargos_1m_vs_6m',
 'share_cp_egresos',
 'share_cp_ingresos',
 'ros_por_trx_3m',
 'ingresos_vs_facturacion',
 'pasivo_vs_ingresos',
 'ratio_egresos_exterior',
 'ratio_ingresos_exterior',
]

# ── Filtrar periodos y columnas existentes ───────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
variables = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas: {no_existentes}")

# Convertir a numérico
for col in variables:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df_eda = df[variables + [TARGET_COL]].copy()

print(f"✓ Shape EDA: {df_eda.shape}")
print(f"✓ Target rate: {df_eda[TARGET_COL].mean():.4f}")


# ════════════════════════════════════════════════════════════════════════════
# 1. RESUMEN ESTADÍSTICO
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("1. RESUMEN ESTADÍSTICO")
print("="*60)

stats = df_eda[variables].describe().T
stats["missing"]     = df_eda[variables].isnull().sum()
stats["missing_pct"] = (df_eda[variables].isnull().mean() * 100).round(2)
stats["zeros_pct"]   = ((df_eda[variables] == 0).mean() * 100).round(2)
stats["skewness"]    = df_eda[variables].skew().round(3)
stats["kurtosis"]    = df_eda[variables].kurt().round(3)

display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max",
               "missing_pct", "zeros_pct", "skewness", "kurtosis"]])

stats.to_csv(os.path.join(OUTPUT_DIR, "01_resumen_estadistico.csv"))
print(f"✓ Guardado: 01_resumen_estadistico.csv")


# ════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("2. MISSING VALUES")
print("="*60)

missing = df_eda[variables].isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

if missing.empty:
    print("✓ No hay missing values")
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing) * 0.35)))
    missing.plot(kind="barh", ax=ax, color="#E07B54")
    ax.set_xlabel("% Missing", fontsize=11)
    ax.set_title("Missing Values por Variable (%)", fontsize=13)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    for i, v in enumerate(missing):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_missing_values.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"✓ Guardado: 02_missing_values.png")
    display(missing.to_frame("missing_pct"))


# ════════════════════════════════════════════════════════════════════════════
# 3. DISTRIBUCIONES (histogramas)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("3. DISTRIBUCIONES")
print("="*60)

n_cols  = 4
n_rows  = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    data = df_eda[col].dropna()
    axes[i].hist(data, bins=40, color="#4C72B0", alpha=0.75, edgecolor="white")
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.1f}")
    ))

# Ocultar ejes vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuciones de Variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_distribuciones.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 03_distribuciones.png")


# ════════════════════════════════════════════════════════════════════════════
# 4. CORRELACIÓN CON EL TARGET
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("4. CORRELACIÓN CON EL TARGET")
print("="*60)

corr_target = (
    df_eda[variables + [TARGET_COL]]
    .corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_target) * 0.35)))
colors = ["#DD4949" if v > 0 else "#4C72B0" for v in corr_target]
corr_target.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson", fontsize=11)
ax.set_title(f"Correlación de Variables con '{TARGET_COL}'", fontsize=13)
for i, v in enumerate(corr_target):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_correlacion_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 04_correlacion_target.png")

display(corr_target.to_frame("corr_con_target").style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
corr_target.to_frame("corr_con_target").to_csv(os.path.join(OUTPUT_DIR, "04_correlacion_target.csv"))


# Reemplaza SOLO la sección 5 - MATRIZ DE CORRELACIÓN
# ════════════════════════════════════════════════════════════════════════════
# 5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES (sin seaborn)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES")
print("="*60)

corr_matrix = df_eda[variables].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_plot = corr_matrix.copy()
corr_plot[mask] = np.nan  # ocultar triángulo superior

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(corr_plot, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.6)

# Etiquetas
ax.set_xticks(range(len(variables)))
ax.set_yticks(range(len(variables)))
ax.set_xticklabels(variables, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(variables, fontsize=8)

# Anotaciones numéricas
for i in range(len(variables)):
    for j in range(len(variables)):
        val = corr_plot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=5.5, color="black" if abs(val) < 0.7 else "white")

ax.set_title("Matriz de Correlación entre Variables", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_matriz_correlacion.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 05_matriz_correlacion.png")

# Pares con alta correlación (> 0.7)
corr_upper = corr_matrix.where(mask == False).stack().reset_index()
corr_upper.columns = ["var1", "var2", "correlacion"]
corr_upper = corr_upper[corr_upper["var1"] != corr_upper["var2"]]
alta_corr  = corr_upper[corr_upper["correlacion"].abs() > 0.7].sort_values("correlacion", key=abs, ascending=False)

print(f"\n⚠ Pares con correlación > 0.7:")
display(alta_corr)
alta_corr.to_csv(os.path.join(OUTPUT_DIR, "05_alta_correlacion.csv"), index=False)


# ════════════════════════════════════════════════════════════════════════════
# 6. BOXPLOT TARGET=0 vs TARGET=1
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("6. DISTRIBUCIÓN POR TARGET (0 vs 1)")
print("="*60)

n_cols = 4
n_rows = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    g0 = df_eda[df_eda[TARGET_COL] == 0][col].dropna()
    g1 = df_eda[df_eda[TARGET_COL] == 1][col].dropna()
    axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
                    patch_artist=True,
                    boxprops=dict(facecolor="#4C72B0", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2))
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución por Target (0 vs 1)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_boxplot_por_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 06_boxplot_por_target.png")


# ════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : {len(variables):<25}║
║  Registros            : {df_eda.shape[0]:<25,}║
║  Target rate          : {df_eda[TARGET_COL].mean():<25.4f}║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")

✓ Shape EDA: (1338493, 37)
✓ Target rate: 0.0008

1. RESUMEN ESTADÍSTICO


,count,mean,std,min,25%,50%,75%,max,missing_pct,zeros_pct,skewness,kurtosis
mto_pas_soles,1338493.00,72227.55,4307705.10,-17.08,0.00,243.38,5267.45,1827472727.91,0.00,26.17,218.91,65226.62
imp_trx_abonosefect_6m,1338493.00,29857.58,585683.21,0.00,0.00,0.00,491.00,178222039.43,0.00,71.72,149.37,31484.33
imp_trx_cargosefe_6m,1338493.00,27350.05,267608.06,0.00,0.00,0.00,2555.12,37468900.00,0.00,67.21,49.23,4038.60
avg_trx_cargostot_3m,1338493.00,0.63,2.13,0.00,0.00,0.00,0.33,157.33,0.00,74.56,8.13,138.57
cnt_trx_cargostot_3m,1338493.00,49.40,477.57,0.00,1.00,11.00,42.00,108903.00,0.00,24.48,124.12,19975.30
cnt_trx_abonospromtot_3m,1338493.00,16.74,1033.32,0.00,0.00,1.00,4.67,453460.67,0.00,31.82,367.24,144801.12
rat_trx_abonosefectot_1m,1338493.00,0.05,0.18,0.00,0.00,0.00,0.00,1.00,0.00,88.74,4.35,18.73
rat_trx_abonosefectot_3m,1338493.00,0.07,0.20,0.00,0.00,0.00,0.00,1.00,0.00,79.88,3.52,11.90
rat_trx_abonosefectot_9m,1338493.00,0.09,0.23,0.00,0.00,0.00,0.05,1.00,0.00,66.02,2.90,7.73
rat_mntcrgsefetot_1m,1338493.00,0.09,0.25,0.00,0.00,0.00,0.00,1.00,0.00,83.56,2.86,6.69


✓ Guardado: 01_resumen_estadistico.csv

2. MISSING VALUES
✓ Guardado: 02_missing_values.png


,missing_pct
cod_sectorista_id,90.13
mto_fact_declarado_sunat,76.79



3. DISTRIBUCIONES
✓ Guardado: 03_distribuciones.png

4. CORRELACIÓN CON EL TARGET
✓ Guardado: 04_correlacion_target.png


...


5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES
✓ Guardado: 05_matriz_correlacion.png

⚠ Pares con correlación > 0.7:


,var1,var2,correlacion
189,max_mto_cpegrmen_12m,avg_cpmenegr_12m,0.84
35,rat_trx_abonosefectot_9m,rat_trx_abonosefectot_3m,0.73



6. DISTRIBUCIÓN POR TARGET (0 vs 1)


C:\Users\b46637\AppData\Local\Temp\ipykernel_24716\703157225.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_24716\703157225.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_24716\703157225.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_24716\703157225.py:249: MatplotlibDeprecationWarning: The 

✓ Guardado: 06_boxplot_por_target.png

╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : 36                       ║
║  Registros            : 1,338,493                ║
║  Target rate          : 0.0008                   ║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\eda_graficos_test



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Rutas ────────────────────────────────────────────────────────────────────
TRAIN_CORR = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_TRAIN\04_correlacion_target.csv"
TEST_CORR  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_test\04_correlacion_target.csv"
TRAIN_STAT = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_TRAIN\01_resumen_estadistico.csv"
TEST_STAT  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_test\01_resumen_estadistico.csv"
TRAIN_ALTA = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_TRAIN\05_alta_correlacion.csv"
TEST_ALTA  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\eda_graficos_test\05_alta_correlacion.csv"
OUTPUT_DIR = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\auditoria_tier4"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# A. COMPARACIÓN CORRELACIONES TRAIN vs TEST
# ════════════════════════════════════════════════════════════════════════════
df_corr_train = pd.read_csv(TRAIN_CORR, index_col=0).rename(columns={"corr_con_target": "corr_train"})
df_corr_test  = pd.read_csv(TEST_CORR,  index_col=0).rename(columns={"corr_con_target": "corr_test"})

df_corr = df_corr_train.join(df_corr_test, how="outer").reset_index()
df_corr.columns = ["variable", "corr_train", "corr_test"]
df_corr["diferencia_abs"]  = (df_corr["corr_train"] - df_corr["corr_test"]).abs()
df_corr["cambio_signo"]    = (np.sign(df_corr["corr_train"]) != np.sign(df_corr["corr_test"]))
df_corr = df_corr.sort_values("diferencia_abs", ascending=False)

print("="*60)
print("A. COMPARACIÓN CORRELACIONES TRAIN vs TEST")
print("="*60)
display(df_corr.style
    .background_gradient(subset=["diferencia_abs"], cmap="Reds")
    .applymap(lambda v: "background-color: #FFCCCC" if v else "", subset=["cambio_signo"])
    .format({"corr_train": "{:.4f}", "corr_test": "{:.4f}", "diferencia_abs": "{:.4f}"})
)

# Variables con cambio de signo (MUY sospechoso para auditoría)
cambios = df_corr[df_corr["cambio_signo"] == True]
if not cambios.empty:
    print(f"\n🔴 ALERTA: Variables con CAMBIO DE SIGNO entre Train y Test:")
    display(cambios)
else:
    print(f"\n✅ No hay cambios de signo entre Train y Test")

# Variables con diferencia > 0.02
grandes_diffs = df_corr[df_corr["diferencia_abs"] > 0.02]
if not grandes_diffs.empty:
    print(f"\n⚠️  Variables con diferencia > 0.02 entre Train y Test:")
    display(grandes_diffs)

df_corr.to_csv(os.path.join(OUTPUT_DIR, "A_correlaciones_train_vs_test.csv"), index=False)

# ════════════════════════════════════════════════════════════════════════════
# B. GRÁFICO COMPARATIVO CORRELACIONES
# ════════════════════════════════════════════════════════════════════════════
df_plot = df_corr.set_index("variable")[["corr_train", "corr_test"]].sort_values("corr_train", key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(12, max(8, len(df_plot) * 0.35)))
x = np.arange(len(df_plot))
width = 0.35
ax.barh(x - width/2, df_plot["corr_train"], width, label="Train", color="#4C72B0", alpha=0.8)
ax.barh(x + width/2, df_plot["corr_test"],  width, label="Test",  color="#DD4949", alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(df_plot.index, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación con Target", fontsize=11)
ax.set_title("Correlación con Target: Train vs Test", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "B_correlacion_train_vs_test.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"\n✓ Guardado: B_correlacion_train_vs_test.png")

# ════════════════════════════════════════════════════════════════════════════
# C. COMPARACIÓN ESTADÍSTICOS DESCRIPTIVOS TRAIN vs TEST
# ════════════════════════════════════════════════════════════════════════════
df_stat_train = pd.read_csv(TRAIN_STAT, index_col=0)
df_stat_test  = pd.read_csv(TEST_STAT,  index_col=0)

# Comparar medias y stds
comparacion_stats = pd.DataFrame({
    "mean_train":     df_stat_train["mean"],
    "mean_test":      df_stat_test["mean"],
    "std_train":      df_stat_train["std"],
    "std_test":       df_stat_test["std"],
    "missing_train":  df_stat_train["missing_pct"],
    "missing_test":   df_stat_test["missing_pct"],
    "zeros_train":    df_stat_train["zeros_pct"],
    "zeros_test":     df_stat_test["zeros_pct"],
})

comparacion_stats["drift_mean_%"] = (
    ((comparacion_stats["mean_test"] - comparacion_stats["mean_train"]) / 
     (comparacion_stats["mean_train"].abs() + 1e-10)) * 100
).round(2)

comparacion_stats["drift_std_%"] = (
    ((comparacion_stats["std_test"] - comparacion_stats["std_train"]) / 
     (comparacion_stats["std_train"].abs() + 1e-10)) * 100
).round(2)

comparacion_stats["diff_missing"] = (comparacion_stats["missing_test"] - comparacion_stats["missing_train"]).round(2)
comparacion_stats["diff_zeros"]   = (comparacion_stats["zeros_test"]   - comparacion_stats["zeros_train"]).round(2)

comparacion_stats = comparacion_stats.sort_values("drift_mean_%", key=abs, ascending=False)

print("\n" + "="*60)
print("C. DRIFT DE ESTADÍSTICOS TRAIN vs TEST")
print("="*60)
display(comparacion_stats.style
    .background_gradient(subset=["drift_mean_%", "drift_std_%"], cmap="RdYlGn_r")
    .format("{:.2f}")
)

# Alertas de drift
drift_alto = comparacion_stats[comparacion_stats["drift_mean_%"].abs() > 50]
if not drift_alto.empty:
    print(f"\n🔴 Variables con drift de media > 50% entre Train y Test:")
    display(drift_alto[["mean_train", "mean_test", "drift_mean_%"]])

comparacion_stats.to_csv(os.path.join(OUTPUT_DIR, "C_drift_estadisticos_train_vs_test.csv"))
print(f"\n✓ Guardado: C_drift_estadisticos_train_vs_test.csv")

# ════════════════════════════════════════════════════════════════════════════
# D. ALTA CORRELACIÓN ENTRE VARIABLES (multicolinealidad)
# ════════════════════════════════════════════════════════════════════════════
df_alta_train = pd.read_csv(TRAIN_ALTA)
df_alta_test  = pd.read_csv(TEST_ALTA)

print("\n" + "="*60)
print("D. MULTICOLINEALIDAD - ALTA CORRELACIÓN ENTRE VARIABLES")
print("="*60)
print(f"\n🔵 Train — pares con correlación > 0.7: {len(df_alta_train)}")
display(df_alta_train)

print(f"\n🔴 Test — pares con correlación > 0.7: {len(df_alta_test)}")
display(df_alta_test)

# Pares que aparecen en train pero no en test (o viceversa)
pares_train = set(zip(df_alta_train["var1"], df_alta_train["var2"]))
pares_test  = set(zip(df_alta_test["var1"],  df_alta_test["var2"]))
solo_train  = pares_train - pares_test
solo_test   = pares_test  - pares_train

if solo_train:
    print(f"\n⚠️  Pares con alta corr SOLO en Train (no en Test):")
    for p in solo_train: print(f"   → {p[0]}  ↔  {p[1]}")
if solo_test:
    print(f"\n⚠️  Pares con alta corr SOLO en Test (no en Train):")
    for p in solo_test: print(f"   → {p[0]}  ↔  {p[1]}")

# ════════════════════════════════════════════════════════════════════════════
# RESUMEN PARA AUDITORÍA
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════════════╗
║         RESUMEN PARA AUDITORÍA TIER 4  ✅               ║
╠══════════════════════════════════════════════════════════╣
║  Archivos generados en:                                  ║
║  A_correlaciones_train_vs_test.csv                       ║
║  B_correlacion_train_vs_test.png                         ║
║  C_drift_estadisticos_train_vs_test.csv                  ║
╚══════════════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")

A. COMPARACIÓN CORRELACIONES TRAIN vs TEST


C:\Users\b46637\AppData\Local\Temp\ipykernel_52900\684162413.py:34: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda v: "background-color: #FFCCCC" if v else "", subset=["cambio_signo"])


,variable,corr_train,corr_test,diferencia_abs,cambio_signo
16,imp_trx_cargosefe_6m,0.2096,0.0858,0.1238,False
30,ratio_cargos_1m_vs_6m,0.1560,0.0365,0.1196,False
15,imp_trx_abonosefect_6m,0.1391,0.0533,0.0858,False
32,ratio_ingresos_exterior,0.1217,0.0440,0.0777,False
12,flg_alerta_12m,0.1206,0.0439,0.0767,False
27,rat_trx_abonosefectot_3m,0.0953,0.0287,0.0666,False
2,avg_trx_cargostot_3m,0.0946,0.0336,0.0610,False
26,rat_trx_abonosefectot_1m,0.0851,0.0268,0.0583,False
28,rat_trx_abonosefectot_9m,0.0828,0.0257,0.0571,False
1,avg_cpmenegr_12m,0.0540,0.0034,0.0505,False



🔴 ALERTA: Variables con CAMBIO DE SIGNO entre Train y Test:


,variable,corr_train,corr_test,diferencia_abs,cambio_signo
4,cnt_meses_sinegresos_12m,0.001048,-0.004694,0.005742,True
17,ingresos_vs_facturacion,-0.000716,0.002108,0.002823,True


...


✓ Guardado: C_drift_estadisticos_train_vs_test.csv

D. MULTICOLINEALIDAD - ALTA CORRELACIÓN ENTRE VARIABLES

🔵 Train — pares con correlación > 0.7: 2


,var1,var2,correlacion
0,rat_trx_abonosefectot_9m,rat_trx_abonosefectot_3m,0.729636
1,rat_trx_abonosefectot_3m,rat_trx_abonosefectot_1m,0.703878



🔴 Test — pares con correlación > 0.7: 2


,var1,var2,correlacion
0,max_mto_cpegrmen_12m,avg_cpmenegr_12m,0.838079
1,rat_trx_abonosefectot_9m,rat_trx_abonosefectot_3m,0.726015



⚠️  Pares con alta corr SOLO en Train (no en Test):
   → rat_trx_abonosefectot_3m  ↔  rat_trx_abonosefectot_1m

⚠️  Pares con alta corr SOLO en Test (no en Train):
   → max_mto_cpegrmen_12m  ↔  avg_cpmenegr_12m

╔══════════════════════════════════════════════════════════╗
║         RESUMEN PARA AUDITORÍA TIER 4  ✅               ║
╠══════════════════════════════════════════════════════════╣
║  Archivos generados en:                                  ║
║  A_correlaciones_train_vs_test.csv                       ║
║  B_correlacion_train_vs_test.png                         ║
║  C_drift_estadisticos_train_vs_test.csv                  ║
╚══════════════════════════════════════════════════════════╝
📁 C:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarrollo\auditoria_tier4

